# Qubit Quest: from intuition to Qiskit
An explainable Fall Fest demo. Run after installing `services/api/requirements.lock.txt`. These are ideal local simulations, not QPU runs.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, partial_trace, SparsePauliOp
from qiskit.circuit.library import QFTGate
from qiskit.primitives import StatevectorSampler
import numpy as np

## Superposition and interference
Predict H|0⟩, then H²|0⟩. Equal measurement probabilities do not imply a classical mixture.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
print('After H:', Statevector(qc).probabilities_dict())
qc.h(0)
print('After HH:', Statevector(qc).probabilities_dict())
assert np.isclose(Statevector(qc).probabilities()[0], 1)

## Bell correlations and reduced states
Bitstrings are q1q0. Tracing out qubit 1 leaves a maximally mixed qubit 0.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
print(Statevector(bell).probabilities_dict())
print(partial_trace(Statevector(bell), [1]))
measured = bell.copy()
measured.measure_all()
print(StatevectorSampler(seed=7).run([measured], shots=1024).result()[0].data.meas.get_counts())

## Grover: mark 11 and amplify
For four candidates and one marked item, one iteration is sufficient. The oracle is supplied, not a free database lookup.

In [ ]:
grover = QuantumCircuit(2)
grover.h([0, 1])
grover.cz(0, 1)
grover.h([0, 1])
grover.x([0, 1])
grover.cz(0, 1)
grover.x([0, 1])
grover.h([0, 1])
print(Statevector(grover).probabilities_dict())
assert np.isclose(Statevector(grover).probabilities()[3], 1)

## QFT: inspect amplitudes, not just histograms

In [ ]:
qft = QuantumCircuit(2)
qft.x(0)
qft.append(QFTGate(2), [0, 1])
print(Statevector(qft).data)
assert np.allclose(Statevector(qft).data, [0.5, 0.5j, -0.5, -0.5j])

## Optimization: one-edge MaxCut
01 and 10 cut the edge. Sweep gamma while fixing beta=-pi/8.

In [ ]:
best = (0, 0)
for gamma in np.linspace(0, 2*np.pi, 101):
    circuit = QuantumCircuit(2)
    circuit.h([0, 1])
    circuit.rzz(gamma, 0, 1)
    circuit.rx(-np.pi/4, [0, 1])
    p = Statevector(circuit).probabilities()
    score = p[1] + p[2]
    if score > best[1]:
        best = (gamma, score)
print('Best angle and expected cut:', best)
assert np.isclose(best[1], 1)

## Chemistry precursor: variational energy
This toy Hamiltonian is −ZI−IZ+0.5XX, in arbitrary units. It is not an ab initio molecule. Real chemistry needs a molecular geometry, basis, active space, mapping, and validated reference energies.

In [ ]:
hamiltonian = SparsePauliOp.from_list([('ZI', -1), ('IZ', -1), ('XX', .5)])
exact = np.linalg.eigvalsh(hamiltonian.to_matrix())[0]
energies = []
for theta in np.linspace(0, 2*np.pi, 201):
    trial = QuantumCircuit(2)
    trial.ry(theta, 0)
    trial.cx(0, 1)
    energies.append(Statevector(trial).expectation_value(hamiltonian).real)
print('Trial minimum:', min(energies), 'Exact ground:', exact)
assert min(energies) >= exact - 1e-10